In [2]:
import copy
import json
import sys
from pathlib import Path

import pandas as pd
import torch
from astropy.io import fits
from sklearn.model_selection import KFold
from torch import nn
from torch.utils.data import Subset
from torchvision import transforms
from torchvision.transforms import Normalize

In [3]:
cwd = Path.cwd()
home_dir = cwd.parent.parent
cutout_dir = home_dir / 'data' / 'cutouts'
log_dir = home_dir / 'logs'
model_dir = cwd / 'models'

In [4]:
sys.path.append(str(home_dir))
import utils as ut

In [5]:
logdf = pd.read_json(log_dir / 'conv.json')
logdf = logdf.sort_values(by='accuracy', ascending=False).head()

params = logdf.iloc[0].to_dict()

In [6]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_FEATURES = ['FLAG']
SEED = params['seed']

# HYPERPARAMETERS
INPUT_SIZE = params['image_size']
EPOCHS = params['epochs']
NUM_TRIALS = params['num_trials']
PATIENCE = params['patience']
SCALER = params['scaling']
N_SPLITS = int(params['N_folds'])
BANDS = params['bands']

In [7]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

if SEED:
    ut.set_seeds(SEED)

## Prepare Data

In [8]:
# LOAD DATA INTO DF

cols = {'idx': [], 'images': [], 'header': []}

for folder in cutout_dir.iterdir():
    if folder.is_dir():
        cols['idx'].append(folder.stem)

        temp_df = ut.fits_df(folder)
        temp_df.sort_values(by='BAND')

        # load images
        temp_imgs = {}
        for _, row in temp_df.iterrows():
            hdul = fits.open(row['FILEPATH'])
            data = hdul[0].data.astype('float32')
            header = hdul[0].header
            temp_imgs[row['BAND']] = data
            hdul.close()
        cols['images'].append(temp_imgs)
        cols['header'].append(header)

# turn dict into df and expand header columns
df = pd.DataFrame(cols)
to_expand = ['header', 'images']
for col in to_expand:
    temp_df = pd.DataFrame(df[col].tolist(), index=df.index)
    df = df.join(temp_df)

df.drop(columns=['header', 'BAND', 'images'], inplace=True)

# shuffle
df = df.sample(frac=1, random_state=SEED) if SEED else df.sample(frac=1)

In [9]:
# STACK AND RESIZE BANDS

resize_transform = transforms.Resize(
    INPUT_SIZE, transforms.InterpolationMode.BILINEAR
)

df['images'] = df[BANDS].values.tolist()
df['images'] = [resize_transform(torch.as_tensor(img)) for img in df['images']]

C:\Users\harty\AppData\Local\Temp\ipykernel_19196\3055811126.py:8: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  df['images'] = [resize_transform(torch.as_tensor(img)) for img in df['images']]


In [10]:
# EXTRACT X, y, and i from data

data_df = df[df['TABLE'] == 'test']

X_data = torch.stack(data_df['images'].tolist())
y_data = torch.tensor(data_df['FLAG'].tolist())
i_data = torch.tensor(data_df['idx'].astype(int).tolist())

In [11]:
X_data.shape

torch.Size([48, 3, 67, 67])

In [12]:
# SCALE DATA


def scale_data(data, type='tanh'):

    if type == 'log1p':
        data = torch.log1p(data)
    elif type == 'pytorch_normalize':
        scaler = Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
        data = scaler(data)
    elif type == 'arcsinh':
        data = torch.asinh(data)
    elif type == 'tanh':
        data = torch.tanh(data)
    else:
        raise ValueError(f'{type} is not a valid scaling function')

    return data


X_data_scaled = scale_data(X_data, type=SCALER)
ut.analyze_data(X_data, axis=(0, 2, 3), name='Unscaled data:')
print('\n')
ut.analyze_data(X_data_scaled, axis=(0, 2, 3), name=f'{SCALER} Scaled data:')

Unscaled data:
Mean:  [0.03735497 0.0975937  0.18473902]
Median:  [0.01111393 0.01988671 0.03066221]
St. Dev:  [0.11759452 0.31093997 0.7222378 ]
Max:  [ 3.7142553  8.909018  31.002878 ]
Min:  [-0.04422804 -0.03231659 -0.16626   ]
Length:  48


tanh Scaled data:
Mean:  [0.03436922 0.07792095 0.11279876]
Median:  [0.01111347 0.01988409 0.0306526 ]
St. Dev:  [0.08343259 0.15521054 0.21317281]
Max:  [0.99881256 0.99999994 1.        ]
Min:  [-0.04419922 -0.03230534 -0.16474481]
Length:  48


In [13]:
# CONVERT TO DATSET

# transforms to apply to inputs
train_transform = transforms.Compose(
    [
        # transforms.RandomHorizontalFlip(p=0.5),
        # transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=(0, 360))
        # transforms.RandomPerspective()
    ]
)

dataset = ut.GalaxyDataset(images=X_data_scaled, ids=i_data, labels=y_data)
test_loader = torch.utils.data.DataLoader(
    dataset, batch_size=16, shuffle=True, num_workers=2
)

## Define Model

In [14]:
# DEFINE MODEL AND HYPERPARAMTER FUNCTIONS


def define_model(n_conv, n_fc, **kwargs):

    conv_channels = [
        params[f'n_channels_conv{layer}'] for layer in range(n_conv)
    ]
    conv_ksizes = [
        params[f'kernel_size_conv{layer}'] for layer in range(n_conv)
    ]
    conv_paddings = [params[f'padding_conv{layer}'] for layer in range(n_conv)]
    conv_dropouts = [params[f'dropout_conv{layer}'] for layer in range(n_conv)]

    fc_units = [params[f'n_units_fc{layer}'] for layer in range(n_fc)]
    fc_dropouts = [params[f'dropout_fc{layer}'] for layer in range(n_fc)]

    layers = []

    # build conv layers
    in_channels = len(BANDS)
    current_spatial_size = INPUT_SIZE

    for i in range(n_conv):
        out_channels = conv_channels[i]
        ksize = conv_ksizes[i]
        pad = conv_paddings[i]
        dropout = conv_dropouts[i]

        layers.extend(
            (
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=ksize,
                    padding=pad,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        )

        current_spatial_size = current_spatial_size - ksize + (2 * pad) + 1
        in_channels = out_channels

    flattened_features = in_channels * (current_spatial_size**2)
    layers.append(nn.Flatten())

    # build fc layers
    in_features = flattened_features
    for i in range(n_fc):
        out_features = fc_units[i]
        dropout = fc_dropouts[i]

        layers.extend(
            (
                nn.Linear(in_features, out_features, bias=False),
                nn.BatchNorm1d(out_features),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        )

        in_features = out_features

    layers.append(nn.Linear(in_features, 2 * len(OUT_FEATURES)))

    return nn.Sequential(*layers)

## Get Predictions

In [15]:
models = []

for fold in range(N_SPLITS):
    model = define_model(**params).to(DEVICE)
    model.load_state_dict(
        torch.load(model_dir / f'fold{fold}.pth', map_location=DEVICE)
    )
    model.to(DEVICE)
    model.eval()
    models.append(model)

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\harty\\Code\\Galaxy-Classifier\\Summer Research\\training\\Convolutional\\models\\fold3.pth'

In [ ]:
def predict_ensemble(models, test_loader, device, task='classification'):
    all_predictions = []

    with torch.no_grad():
        for inputs in test_loader:
            # If test_loader returns (inputs, labels), unpack inputs: inputs = inputs[0]
            inputs = inputs.to(device)
            batch_preds = []

            for model in models:
                outputs = model(inputs)

                if task == 'classification':
                    # For multi-class classification:
                    probs = torch.softmax(outputs, dim=1)
                    # For binary classification with 1 output node, use:
                    # probs = torch.sigmoid(outputs)
                    batch_preds.append(probs)
                else:
                    # For regression, use raw outputs
                    batch_preds.append(outputs)

            # Stack predictions along a new dimension: shape (k_folds, batch_size, num_classes)
            stacked_preds = torch.stack(batch_preds, dim=0)

            # Average across the folds (dim=0): shape (batch_size, num_classes)
            avg_preds = torch.mean(stacked_preds, dim=0)

            if task == 'classification':
                # Get the class index with the highest average probability
                final_batch_preds = torch.argmax(avg_preds, dim=1)
                # For binary classification, use: (avg_preds > 0.5).int()
            else:
                final_batch_preds = avg_preds

            all_predictions.append(final_batch_preds.cpu())

    # Concatenate all batches into a single tensor
    return torch.cat(all_predictions, dim=0)


In [ ]:
predictions = predict_ensemble(
    models, test_loader, DEVICE, task='classification'
)
print(predictions.shape)


NameError: name 'predict_ensemble' is not defined